In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Dowload Dataset

In [6]:
import kagglehub

# Prompts for your Kaggle username and key from kaggle.json
kagglehub.login()



Kaggle credentials set.
Kaggle credentials successfully validated.


In [8]:
path = kagglehub.competition_download('h-and-m-personalized-fashion-recommendations')
print("Path to competition files:", path)

100%|██████████| 28.7G/28.7G [03:46<00:00, 136MB/s]


Extracting files...
Path to competition files: /root/.cache/kagglehub/competitions/h-and-m-personalized-fashion-recommendations


In [10]:
from pathlib import Path
import shutil

src = Path(
    "/root/.cache/kagglehub/competitions/"
    "h-and-m-personalized-fashion-recommendations"
)
dst = Path("/content/data")

# 确认源目录存在
assert src.exists(), f"Source not found: {src}"

# 创建目标目录
dst.mkdir(parents=True, exist_ok=True)

# 移动全部文件及文件夹（包括 images）
for item in src.iterdir():
    target = dst / item.name

    if target.exists():
        print(f"Skipping existing: {target}")
        continue

    shutil.move(str(item), str(target))
    print(f"Moved: {item.name}")

print("\n✅ Dataset moved to:", dst)

Moved: sample_submission.csv
Moved: images
Moved: transactions_train.csv
Moved: customers.csv
Moved: articles.csv

✅ Dataset moved to: /content/data


In [11]:
print("Files in /content/data:")
for item in sorted(dst.iterdir()):
    print(f"{'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

# 验证主要文件
for filename in [
    "articles.csv",
    "customers.csv",
    "transactions_train.csv"
]:
    assert (dst / filename).exists(), f"Missing: {filename}"

print("\n✅ H&M core dataset files verified!")

Files in /content/data:
[FILE] articles.csv
[FILE] customers.csv
[DIR] images
[FILE] sample_submission.csv
[FILE] transactions_train.csv

✅ H&M core dataset files verified!


In [16]:
from pathlib import Path
import shutil
import subprocess

source_dir = Path("/content/data")
target_dir = Path(
    "/content/drive/MyDrive/projects/e_commerce_recsys/data/original data"
)

archive_path = source_dir / "images.tar"
target_dir.mkdir(parents=True, exist_ok=True)

# Create an uncompressed image archive
subprocess.run(
    ["tar", "-cf", str(archive_path), "-C", str(source_dir), "images"],
    check=True
)

# Copy the archive to Google Drive
shutil.copy2(archive_path, target_dir / archive_path.name)

print("Archive copied to Google Drive.")

Archive copied to Google Drive.


In [17]:
import hashlib

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

destination = target_dir / "images.tar"

assert archive_path.stat().st_size == destination.stat().st_size
assert sha256_file(archive_path) == sha256_file(destination)

print("Archive integrity verified.")

Archive integrity verified.


In [18]:
from pathlib import Path
import shutil
import hashlib

source_dir = Path("/content/data")
target_dir = Path(
    "/content/drive/MyDrive/projects/e_commerce_recsys/data/original data"
)

csv_files = [
    "articles.csv",
    "customers.csv",
    "transactions_train.csv",
    "sample_submission.csv",
]

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

target_dir.mkdir(parents=True, exist_ok=True)

for filename in csv_files:
    source = source_dir / filename
    destination = target_dir / filename

    assert source.is_file(), f"Source file not found: {source}"

    print(f"Copying {filename}...")
    shutil.copy2(source, destination)

    assert source.stat().st_size == destination.stat().st_size
    assert sha256_file(source) == sha256_file(destination), (
        f"Checksum mismatch: {filename}"
    )

    print(f"Verified: {filename}")

print("\nAll CSV files copied and verified successfully.")

Copying articles.csv...
Verified: articles.csv
Copying customers.csv...
Verified: customers.csv
Copying transactions_train.csv...
Verified: transactions_train.csv
Copying sample_submission.csv...
Verified: sample_submission.csv

All CSV files copied and verified successfully.


# H&M Personalized Fashion — Phase 0 Data Quality & EDA

**Goal:** Auditable, aggregate-only EDA for `articles.csv`, `customers.csv`, `transactions_train.csv`, and the local product-image inventory. No modeling, final chronological split, image download, or feature-store decisions.

**Privacy:** Do not publish raw customer identifiers, postal codes, or row-level transaction exports. All outputs are aggregate summaries. A local Colab runtime is ephemeral.

**Image coverage policy:** Only calculate exact presence rates after confirming that the entire image filename inventory has been enumerated from a complete source. Filename presence is **not** evidence of image decodability or usable embeddings.

## 0. Runtime configuration

In [19]:
%pip -q install "duckdb>=1.1" "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "pillow>=10"

In [20]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import zipfile
from datetime import datetime, timedelta

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('/content/data')
OUTPUT_DIR = Path('/content/eda_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to True only after confirming that the entire extracted Kaggle image set is present.
# A directory existing on disk does not independently prove completeness.
CONFIRM_LOCAL_IMAGE_INVENTORY_COMPLETE = False

# Optional: point to an already-downloaded complete Kaggle archive containing images/.
IMAGE_ARCHIVE = None  # Example: Path('/content/hm_competition.zip')

# Do not download images in this notebook.
DOWNLOAD_IMAGES = False
assert DOWNLOAD_IMAGES is False, 'Image downloads are intentionally disabled in Phase 0.'

FILES = {name: DATA_DIR / f'{name}.csv' for name in ('articles', 'customers', 'transactions_train')}
for name, path in FILES.items():
    assert path.is_file(), f'Missing input: {path}'

con = duckdb.connect(database=':memory:')
con.execute('PRAGMA threads=4')
con.execute("PRAGMA memory_limit='12GB'")  # Adjust for the selected Colab runtime.

def sql_string(path):
    return "'" + str(path).replace("'", "''") + "'"

def report(name, frame, max_rows=20):
    result = frame.copy()
    result.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)
    print(f'[{name}] {len(result):,} aggregate rows; saved to {OUTPUT_DIR / (name + ".csv")}')
    display(result.head(max_rows))
    return result

def query(sql, params=None):
    return con.execute(sql, params or []).df()

print('Input directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)
for name, path in FILES.items():
    print(f'{name}: {path.stat().st_size / 2**30:.3f} GiB')

Input directory: /content/data
Output directory: /content/eda_outputs
articles: 0.034 GiB
customers: 0.193 GiB
transactions_train: 3.248 GiB


### Canonical CSV views

`article_id` and `customer_id` stay strings. Leading zeros in article IDs must never be dropped. `TRY_CAST` is used for date and numeric diagnostics, not to silently discard malformed values.

In [21]:
con.execute(f"""
CREATE VIEW articles_raw AS
SELECT * FROM read_csv_auto({sql_string(FILES['articles'])},
    header=true, all_varchar=true, sample_size=-1)
""")
con.execute(f"""
CREATE VIEW customers_raw AS
SELECT * FROM read_csv_auto({sql_string(FILES['customers'])},
    header=true, all_varchar=true, sample_size=-1)
""")
con.execute(f"""
CREATE VIEW transactions_raw AS
SELECT * FROM read_csv_auto({sql_string(FILES['transactions_train'])},
    header=true, all_varchar=true, sample_size=-1)
""")
con.execute('''CREATE VIEW transactions AS
SELECT *, TRY_CAST(t_dat AS DATE) AS purchase_date,
          TRY_CAST(price AS DOUBLE) AS price_num
FROM transactions_raw''')

for table in ('articles_raw', 'customers_raw', 'transactions_raw'):
    report(f'{table}_schema', query(f'DESCRIBE SELECT * FROM {table}'), 100)


[articles_raw_schema] 25 aggregate rows; saved to /content/eda_outputs/articles_raw_schema.csv


,column_name,column_type,null,key,default,extra
0,article_id,VARCHAR,YES,None,None,None
1,product_code,VARCHAR,YES,None,None,None
2,prod_name,VARCHAR,YES,None,None,None
3,product_type_no,VARCHAR,YES,None,None,None
4,product_type_name,VARCHAR,YES,None,None,None
5,product_group_name,VARCHAR,YES,None,None,None
6,graphical_appearance_no,VARCHAR,YES,None,None,None
7,graphical_appearance_name,VARCHAR,YES,None,None,None
8,colour_group_code,VARCHAR,YES,None,None,None
9,colour_group_name,VARCHAR,YES,None,None,None


[customers_raw_schema] 7 aggregate rows; saved to /content/eda_outputs/customers_raw_schema.csv


,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,FN,VARCHAR,YES,None,None,None
2,Active,VARCHAR,YES,None,None,None
3,club_member_status,VARCHAR,YES,None,None,None
4,fashion_news_frequency,VARCHAR,YES,None,None,None
5,age,VARCHAR,YES,None,None,None
6,postal_code,VARCHAR,YES,None,None,None


[transactions_raw_schema] 5 aggregate rows; saved to /content/eda_outputs/transactions_raw_schema.csv


,column_name,column_type,null,key,default,extra
0,t_dat,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,article_id,VARCHAR,YES,None,None,None
3,price,VARCHAR,YES,None,None,None
4,sales_channel_id,VARCHAR,YES,None,None,None


## 1. Article catalog quality

In [23]:
## 1. Article catalog quality

article_overview = report('article_overview', query("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT article_id) AS unique_article_ids,
    COUNT(*) FILTER (
        WHERE article_id IS NULL OR TRIM(article_id) = ''
    ) AS null_or_blank_ids,
    COUNT(*) - COUNT(DISTINCT article_id) AS nonunique_id_rows,
    (
        SELECT COUNT(*) FROM articles_raw
    ) - (
        SELECT COUNT(*)
        FROM (SELECT DISTINCT * FROM articles_raw)
    ) AS exact_duplicate_rows
FROM articles_raw
"""))

# Identify duplicate article IDs and conflicting records
article_conflicts = report('article_duplicate_id_diagnostics', query("""
WITH duplicate_ids AS (
    SELECT
        article_id,
        COUNT(*) AS n_rows
    FROM articles_raw
    WHERE article_id IS NOT NULL
      AND TRIM(article_id) <> ''
    GROUP BY article_id
    HAVING COUNT(*) > 1
),
distinct_variants AS (
    SELECT
        article_id,
        COUNT(*) AS distinct_row_variants
    FROM (
        SELECT DISTINCT *
        FROM articles_raw
        WHERE article_id IS NOT NULL
          AND TRIM(article_id) <> ''
    )
    GROUP BY article_id
)
SELECT
    COUNT(*) AS duplicate_id_count,
    COALESCE(SUM(d.n_rows - 1), 0) AS extra_rows_for_duplicate_ids,
    COUNT(*) FILTER (
        WHERE v.distinct_row_variants > 1
    ) AS conflicting_ids
FROM duplicate_ids d
LEFT JOIN distinct_variants v
    ON d.article_id = v.article_id
"""))

# Calculate column-level missingness
article_cols = query(
    "DESCRIBE SELECT * FROM articles_raw"
)["column_name"].tolist()

article_missing_sql = " UNION ALL ".join(
    f"""
    SELECT
        '{c.replace("'", "''")}' AS column_name,
        COUNT(*) AS rows,
        COUNT(*) FILTER (
            WHERE "{c.replace('"', '""')}" IS NULL
               OR TRIM("{c.replace('"', '""')}") = ''
        ) AS missing_rows
    FROM articles_raw
    """
    for c in article_cols
)

article_missing = query(f"""
SELECT
    *,
    ROUND(
        100.0 * missing_rows / NULLIF(rows, 0),
        3
    ) AS missing_pct
FROM ({article_missing_sql})
ORDER BY missing_pct DESC, column_name
""")

report('article_missingness', article_missing, 100)

[article_overview] 1 aggregate rows; saved to /content/eda_outputs/article_overview.csv


,rows,unique_article_ids,null_or_blank_ids,nonunique_id_rows,exact_duplicate_rows
0,105542,105542,0,0,0


[article_duplicate_id_diagnostics] 1 aggregate rows; saved to /content/eda_outputs/article_duplicate_id_diagnostics.csv


,duplicate_id_count,extra_rows_for_duplicate_ids,conflicting_ids
0,0,0.0,0


[article_missingness] 25 aggregate rows; saved to /content/eda_outputs/article_missingness.csv


,column_name,rows,missing_rows,missing_pct
0,detail_desc,105542,416,0.394
1,article_id,105542,0,0.000
2,colour_group_code,105542,0,0.000
3,colour_group_name,105542,0,0.000
4,department_name,105542,0,0.000
5,department_no,105542,0,0.000
6,garment_group_name,105542,0,0.000
7,garment_group_no,105542,0,0.000
8,graphical_appearance_name,105542,0,0.000
9,graphical_appearance_no,105542,0,0.000


,column_name,rows,missing_rows,missing_pct
0,detail_desc,105542,416,0.394
1,article_id,105542,0,0.000
2,colour_group_code,105542,0,0.000
3,colour_group_name,105542,0,0.000
4,department_name,105542,0,0.000
5,department_no,105542,0,0.000
6,garment_group_name,105542,0,0.000
7,garment_group_no,105542,0,0.000
8,graphical_appearance_name,105542,0,0.000
9,graphical_appearance_no,105542,0,0.000


In [24]:
taxonomy_cols = [c for c in (
    'product_type_name', 'product_group_name', 'section_name',
    'garment_group_name', 'colour_group_name', 'perceived_colour_value_name',
    'graphical_appearance_name', 'index_group_name'
) if c in article_cols]

for col in taxonomy_cols:
    qcol = '"' + col.replace('"', '""') + '"'
    report('article_taxonomy_' + col, query(f"""
        SELECT {qcol} AS category, COUNT(*) AS articles
        FROM articles_raw GROUP BY 1 ORDER BY articles DESC, category
    """), 12)

report('article_product_code_variants', query("""
WITH style AS (
    SELECT product_code,
           COUNT(DISTINCT article_id) AS article_variants,
           COUNT(*) AS article_rows
    FROM articles_raw
    WHERE product_code IS NOT NULL AND TRIM(product_code) <> ''
    GROUP BY product_code
)
SELECT article_variants, COUNT(*) AS product_codes,
       SUM(article_rows) AS article_rows
FROM style GROUP BY 1 ORDER BY article_variants
"""), 30)

report('article_id_format', query("""
SELECT LENGTH(article_id) AS id_length, COUNT(*) AS articles,
       COUNT(*) FILTER (WHERE NOT REGEXP_MATCHES(article_id, '^[0-9]+$')) AS non_digit_ids
FROM articles_raw WHERE article_id IS NOT NULL
GROUP BY 1 ORDER BY 1
"""))

print('Article summary:', article_overview.to_dict('records')[0], article_conflicts.to_dict('records')[0])

[article_taxonomy_product_type_name] 131 aggregate rows; saved to /content/eda_outputs/article_taxonomy_product_type_name.csv


,category,articles
0,Trousers,11169
1,Dress,10362
2,Sweater,9302
3,T-shirt,7904
4,Top,4155
5,Blouse,3979
6,Jacket,3940
7,Shorts,3939
8,Shirt,3405
9,Vest top,2991


[article_taxonomy_product_group_name] 19 aggregate rows; saved to /content/eda_outputs/article_taxonomy_product_group_name.csv


,category,articles
0,Garment Upper body,42741
1,Garment Lower body,19812
2,Garment Full body,13292
3,Accessories,11158
4,Underwear,5490
5,Shoes,5283
6,Swimwear,3127
7,Socks & Tights,2442
8,Nightwear,1899
9,Unknown,121


[article_taxonomy_section_name] 56 aggregate rows; saved to /content/eda_outputs/article_taxonomy_section_name.csv


,category,articles
0,Womens Everyday Collection,7295
1,Divided Collection,7124
2,Baby Essentials & Complements,4932
3,Kids Girl,4469
4,Young Girl,3899
5,Womens Lingerie,3598
6,Girls Underwear & Basics,3490
7,Womens Tailoring,3376
8,Kids Boy,3328
9,Womens Small accessories,3270


... (11 outputs left out)

,article_variants,product_codes,article_rows
0,1,24066,24066.0
1,2,11921,23842.0
2,3,4947,14841.0
3,4,2376,9504.0
4,5,1189,5945.0
5,6,778,4668.0
6,7,459,3213.0
7,8,319,2552.0
8,9,206,1854.0
9,10,180,1800.0


[article_id_format] 1 aggregate rows; saved to /content/eda_outputs/article_id_format.csv


,id_length,articles,non_digit_ids
0,10,105542,0


Article summary: {'rows': 105542, 'unique_article_ids': 105542, 'null_or_blank_ids': 0, 'nonunique_id_rows': 0, 'exact_duplicate_rows': 0} {'duplicate_id_count': 0, 'extra_rows_for_duplicate_ids': 0.0, 'conflicting_ids': 0}


## 2. Customer quality and privacy-safe aggregates

In [25]:
customer_overview = report('customer_overview', query("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT customer_id) AS unique_customer_ids,
       COUNT(*) FILTER (WHERE customer_id IS NULL OR TRIM(customer_id)='') AS null_or_blank_ids,
       COUNT(*) - COUNT(DISTINCT customer_id) AS nonunique_id_rows,
       (SELECT COUNT(*) FROM customers_raw) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM customers_raw)) AS exact_duplicate_rows
FROM customers_raw
"""))

customer_cols = query('DESCRIBE SELECT * FROM customers_raw')['column_name'].tolist()
missing_expr = ', '.join(
    "COUNT(*) FILTER (WHERE \"{}\" IS NULL OR TRIM(\"{}\") = '') AS \"{}\"".format(col, col, col) for col in customer_cols
)
customer_missing_wide = query(f'SELECT COUNT(*) AS rows, {missing_expr} FROM customers_raw')
customer_missing = pd.DataFrame({
    'column_name': customer_cols,
    'missing_rows': [int(customer_missing_wide.at[0, c]) for c in customer_cols],
    'rows': [int(customer_missing_wide.at[0, 'rows'])] * len(customer_cols)
})
customer_missing['missing_pct'] = 100 * customer_missing.missing_rows / customer_missing.rows
report('customer_missingness', customer_missing.sort_values('missing_pct', ascending=False), 100)

report('customer_age_distribution', query("""
SELECT CASE
  WHEN age IS NULL OR TRIM(age)='' THEN 'MISSING'
  WHEN TRY_CAST(age AS INTEGER) IS NULL THEN 'INVALID'
  WHEN TRY_CAST(age AS INTEGER) < 13 THEN 'UNDER_13'
  WHEN TRY_CAST(age AS INTEGER) > 100 THEN 'OVER_100'
  WHEN TRY_CAST(age AS INTEGER) < 20 THEN '13-19'
  WHEN TRY_CAST(age AS INTEGER) < 30 THEN '20-29'
  WHEN TRY_CAST(age AS INTEGER) < 40 THEN '30-39'
  WHEN TRY_CAST(age AS INTEGER) < 50 THEN '40-49'
  WHEN TRY_CAST(age AS INTEGER) < 60 THEN '50-59'
  ELSE '60-100' END AS age_band,
  COUNT(*) AS customers
FROM customers_raw GROUP BY 1 ORDER BY age_band
"""))

for col in ('club_member_status', 'fashion_news_frequency', 'Active'):
    if col in customer_cols:
        report('customer_' + col, query(f'SELECT "{col}" AS category, COUNT(*) AS customers FROM customers_raw GROUP BY 1 ORDER BY customers DESC'), 25)

if 'postal_code' in customer_cols:
    report('customer_postal_code_cardinality', query("""
        SELECT COUNT(DISTINCT postal_code) AS distinct_anonymized_postal_codes,
               COUNT(*) FILTER (WHERE postal_code IS NULL OR TRIM(postal_code)='') AS missing_postal_codes
        FROM customers_raw
    """))


[customer_overview] 1 aggregate rows; saved to /content/eda_outputs/customer_overview.csv


,rows,unique_customer_ids,null_or_blank_ids,nonunique_id_rows,exact_duplicate_rows
0,1371980,1371980,0,0,0


[customer_missingness] 7 aggregate rows; saved to /content/eda_outputs/customer_missingness.csv


,column_name,missing_rows,rows,missing_pct
2,Active,907576,1371980,66.150819
1,FN,895050,1371980,65.237831
4,fashion_news_frequency,16009,1371980,1.166854
5,age,15861,1371980,1.156066
3,club_member_status,6062,1371980,0.441843
0,customer_id,0,1371980,0.000000
6,postal_code,0,1371980,0.000000


[customer_age_distribution] 7 aggregate rows; saved to /content/eda_outputs/customer_age_distribution.csv


,age_band,customers
0,13-19,71583
1,20-29,528358
2,30-39,234068
3,40-49,204118
4,50-59,226242
5,60-100,91750
6,MISSING,15861


... (4 outputs left out)

[customer_Active] 2 aggregate rows; saved to /content/eda_outputs/customer_Active.csv


,category,customers
0,None,907576
1,1.0,464404


[customer_postal_code_cardinality] 1 aggregate rows; saved to /content/eda_outputs/customer_postal_code_cardinality.csv


,distinct_anonymized_postal_codes,missing_postal_codes
0,352899,0


## 3. Transaction integrity, temporal behavior, and implicit-feedback semantics

In [26]:
transaction_overview = report('transaction_overview', query("""
SELECT COUNT(*) AS purchase_lines,
       MIN(purchase_date) AS first_valid_date,
       MAX(purchase_date) AS last_valid_date,
       COUNT(*) FILTER (WHERE purchase_date IS NULL) AS malformed_or_missing_dates,
       COUNT(DISTINCT customer_id) AS distinct_customers,
       COUNT(DISTINCT article_id) AS distinct_articles,
       COUNT(DISTINCT (customer_id, article_id)) AS distinct_customer_article_pairs,
       COUNT(DISTINCT (customer_id, article_id, purchase_date)) AS distinct_customer_article_day_keys,
       COUNT(*) - COUNT(DISTINCT HASH(t_dat, customer_id, article_id, price, sales_channel_id)) AS exact_duplicate_rows
FROM transactions
"""))

report('transaction_monthly', query("""
SELECT DATE_TRUNC('month', purchase_date)::DATE AS month,
       COUNT(*) AS purchase_lines, COUNT(DISTINCT customer_id) AS buyers,
       COUNT(DISTINCT article_id) AS articles
FROM transactions WHERE purchase_date IS NOT NULL
GROUP BY 1 ORDER BY 1
"""), 40)
report('transaction_weekly', query("""
SELECT DATE_TRUNC('week', purchase_date)::DATE AS week_start,
       COUNT(*) AS purchase_lines, COUNT(DISTINCT customer_id) AS buyers
FROM transactions WHERE purchase_date IS NOT NULL
GROUP BY 1 ORDER BY 1
"""), 20)

report('transaction_repeat_keys', query("""
WITH pairs AS (
 SELECT customer_id, article_id, COUNT(*) AS lines
 FROM transactions_raw GROUP BY 1,2
), days AS (
 SELECT customer_id, article_id, t_dat, COUNT(*) AS lines
 FROM transactions_raw GROUP BY 1,2,3
)
SELECT (SELECT COUNT(*) FROM pairs WHERE lines > 1) AS repeating_customer_article_pairs,
       (SELECT COALESCE(SUM(lines - 1),0) FROM pairs WHERE lines > 1) AS extra_lines_on_repeated_pairs,
       (SELECT COUNT(*) FROM days WHERE lines > 1) AS repeated_customer_article_date_keys,
       (SELECT COALESCE(SUM(lines - 1),0) FROM days WHERE lines > 1) AS extra_lines_on_repeated_day_keys
"""))

for entity, key in (('buyer', 'customer_id'), ('article', 'article_id')):
    report(f'transaction_lines_per_{entity}', query(f"""
      WITH degree AS (
        SELECT {key}, COUNT(*) AS n
        FROM transactions_raw WHERE {key} IS NOT NULL
        GROUP BY {key}
      )
      SELECT COUNT(*) AS entities,
             MIN(n) AS min_lines,
             APPROX_QUANTILE(n, 0.5) AS p50,
             APPROX_QUANTILE(n, 0.9) AS p90,
             APPROX_QUANTILE(n, 0.99) AS p99,
             APPROX_QUANTILE(n, 0.999) AS p999,
             MAX(n) AS max_lines,
             AVG(n) AS mean_lines
      FROM degree
    """))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[transaction_overview] 1 aggregate rows; saved to /content/eda_outputs/transaction_overview.csv


,purchase_lines,first_valid_date,last_valid_date,malformed_or_missing_dates,distinct_customers,distinct_articles,distinct_customer_article_pairs,distinct_customer_article_day_keys,exact_duplicate_rows
0,31788324,2018-09-20,2020-09-22,0,1362281,104547,27306439,28575395,2974905


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[transaction_monthly] 25 aggregate rows; saved to /content/eda_outputs/transaction_monthly.csv


,month,purchase_lines,buyers,articles
0,2018-09-01,594776,140340,25698
1,2018-10-01,1397040,274220,33376
2,2018-11-01,1270619,274207,31807
3,2018-12-01,1148827,254059,32618
4,2019-01-01,1263471,242024,32309
5,2019-02-01,1152412,238992,32270
6,2019-03-01,1286750,244166,31749
7,2019-04-01,1476454,260875,33136
8,2019-05-01,1560319,271554,34449
9,2019-06-01,1906202,305190,33346


... (7 outputs left out)

[transaction_lines_per_buyer] 1 aggregate rows; saved to /content/eda_outputs/transaction_lines_per_buyer.csv


,entities,min_lines,p50,p90,p99,p999,max_lines,mean_lines
0,1362281,1,9,60,187,393,1895,23.334631


[transaction_lines_per_article] 1 aggregate rows; saved to /content/eda_outputs/transaction_lines_per_article.csv


,entities,min_lines,p50,p90,p99,p999,max_lines,mean_lines
0,104547,1,65,797,3220,9621,50287,304.057735


In [27]:
report('transaction_price_quality', query("""
SELECT COUNT(*) AS lines,
       COUNT(*) FILTER (WHERE price IS NULL OR TRIM(price)='') AS null_or_blank_prices,
       COUNT(*) FILTER (WHERE price IS NOT NULL AND TRIM(price)<>'' AND price_num IS NULL) AS invalid_numeric_prices,
       COUNT(*) FILTER (WHERE NOT ISFINITE(price_num)) AS nonfinite_prices,
       COUNT(*) FILTER (WHERE price_num < 0) AS negative_prices,
       COUNT(*) FILTER (WHERE price_num = 0) AS zero_prices,
       MIN(price_num) FILTER (WHERE ISFINITE(price_num)) AS min_price,
       APPROX_QUANTILE(price_num, 0.5) FILTER (WHERE ISFINITE(price_num)) AS median_price,
       APPROX_QUANTILE(price_num, 0.99) FILTER (WHERE ISFINITE(price_num)) AS p99_price,
       MAX(price_num) FILTER (WHERE ISFINITE(price_num)) AS max_price
FROM transactions
"""))
report('transaction_sales_channels', query("""
SELECT sales_channel_id AS channel, COUNT(*) AS purchase_lines
FROM transactions_raw GROUP BY 1 ORDER BY purchase_lines DESC
"""))

report('transaction_referential_integrity', query("""
WITH customer_keys AS (SELECT DISTINCT customer_id FROM customers_raw),
     article_keys AS (SELECT DISTINCT article_id FROM articles_raw)
SELECT COUNT(*) AS purchase_lines,
       COUNT(*) FILTER (WHERE c.customer_id IS NULL) AS unmatched_customer_lines,
       COUNT(*) FILTER (WHERE a.article_id IS NULL) AS unmatched_article_lines,
       ROUND(100.0 * COUNT(*) FILTER (WHERE c.customer_id IS NOT NULL) / NULLIF(COUNT(*),0), 4) AS customer_join_coverage_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE a.article_id IS NOT NULL) / NULLIF(COUNT(*),0), 4) AS article_join_coverage_pct,
       COUNT(DISTINCT CASE WHEN c.customer_id IS NULL THEN t.customer_id END) AS distinct_unmatched_customer_ids,
       COUNT(DISTINCT CASE WHEN a.article_id IS NULL THEN t.article_id END) AS distinct_unmatched_article_ids
FROM transactions_raw t
LEFT JOIN customer_keys c ON t.customer_id = c.customer_id
LEFT JOIN article_keys a ON t.article_id = a.article_id
"""))
print('Price values are unitless in this analysis; no real-world currency is assumed.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[transaction_price_quality] 1 aggregate rows; saved to /content/eda_outputs/transaction_price_quality.csv


,lines,null_or_blank_prices,invalid_numeric_prices,nonfinite_prices,negative_prices,zero_prices,min_price,median_price,p99_price,max_price
0,31788324,0,0,0,0,0,0.000017,0.025364,0.096495,0.591525


[transaction_sales_channels] 2 aggregate rows; saved to /content/eda_outputs/transaction_sales_channels.csv


,channel,purchase_lines
0,2,22379862
1,1,9408462


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[transaction_referential_integrity] 1 aggregate rows; saved to /content/eda_outputs/transaction_referential_integrity.csv


,purchase_lines,unmatched_customer_lines,unmatched_article_lines,customer_join_coverage_pct,article_join_coverage_pct,distinct_unmatched_customer_ids,distinct_unmatched_article_ids
0,31788324,0,0,100.0,100.0,0,0


Price values are unitless in this analysis; no real-world currency is assumed.


### Historical activity and a **diagnostic-only** future seven-day window

The last observed seven days are **not** a frozen test split. This diagnostic only checks whether a chronological next-seven-day evaluation could have a meaningful population. No training or label leakage is introduced into model artifacts because none are created here.

In [28]:
report('transaction_repeat_buyer_fraction', query("""
WITH buyers AS (
    SELECT customer_id, COUNT(DISTINCT purchase_date) AS active_days,
           COUNT(*) AS lines
    FROM transactions WHERE purchase_date IS NOT NULL
    GROUP BY 1
)
SELECT COUNT(*) AS buyers,
       COUNT(*) FILTER (WHERE active_days > 1) AS buyers_on_multiple_days,
       ROUND(100.0 * COUNT(*) FILTER (WHERE active_days > 1) / NULLIF(COUNT(*),0), 3) AS repeat_buyer_pct,
       COUNT(*) FILTER (WHERE lines >= 10) AS buyers_with_10plus_lines,
       APPROX_QUANTILE(lines, 0.5) AS median_lines,
       APPROX_QUANTILE(lines, 0.9) AS p90_lines
FROM buyers
"""))

report('future_7day_feasibility_diagnostic', query("""
WITH bound AS (SELECT MAX(purchase_date) AS max_date FROM transactions),
     history AS (
         SELECT customer_id, COUNT(*) AS historical_lines,
                COUNT(DISTINCT article_id) AS historical_distinct_articles
         FROM transactions, bound
         WHERE purchase_date < bound.max_date - INTERVAL 6 DAY
         GROUP BY 1
     ),
     future AS (
         SELECT customer_id, COUNT(*) AS future_lines,
                COUNT(DISTINCT article_id) AS future_distinct_articles
         FROM transactions, bound
         WHERE purchase_date BETWEEN bound.max_date - INTERVAL 6 DAY AND bound.max_date
         GROUP BY 1
     )
SELECT (SELECT max_date - INTERVAL 6 DAY FROM bound)::DATE AS diagnostic_start,
       (SELECT max_date FROM bound) AS diagnostic_end,
       COUNT(*) AS buyers_with_future_purchases,
       COUNT(*) FILTER (WHERE h.customer_id IS NOT NULL) AS returning_buyers,
       COUNT(*) FILTER (WHERE h.historical_lines >= 10) AS returning_buyers_10plus_history,
       SUM(f.future_lines) AS future_purchase_lines,
       APPROX_QUANTILE(h.historical_lines, 0.5) FILTER (WHERE h.historical_lines IS NOT NULL) AS returning_history_p50,
       APPROX_QUANTILE(h.historical_lines, 0.9) FILTER (WHERE h.historical_lines IS NOT NULL) AS returning_history_p90
FROM future f LEFT JOIN history h USING (customer_id)
"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[transaction_repeat_buyer_fraction] 1 aggregate rows; saved to /content/eda_outputs/transaction_repeat_buyer_fraction.csv


,buyers,buyers_on_multiple_days,repeat_buyer_pct,buyers_with_10plus_lines,median_lines,p90_lines
0,1362281,914308,67.116,676044,9,60


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[future_7day_feasibility_diagnostic] 1 aggregate rows; saved to /content/eda_outputs/future_7day_feasibility_diagnostic.csv


,diagnostic_start,diagnostic_end,buyers_with_future_purchases,returning_buyers,returning_buyers_10plus_history,future_purchase_lines,returning_history_p50,returning_history_p90
0,2020-09-16,2020-09-22,68984,63412,54242,240311.0,40,148


,diagnostic_start,diagnostic_end,buyers_with_future_purchases,returning_buyers,returning_buyers_10plus_history,future_purchase_lines,returning_history_p50,returning_history_p90
0,2020-09-16,2020-09-22,68984,63412,54242,240311.0,40,148


## 4. Image inventory completeness and coverage (priority)

**Decision tree:**

1. A verified complete ZIP inventory (central directory) can be compared with the fully extracted local tree.
2. Otherwise, an already downloaded local tree is accepted **only with explicit user attestation** that the official competition extraction finished without error and all image folders were transferred. This is *user-attested completeness*, not independently authenticated official completeness.
3. Otherwise `image_coverage_status = 'NOT_MEASURED'`. Do not infer file presence from predicted image paths.

The Kaggle CLI/API could expose paginated file listings, but capabilities vary by version. The next cell inspects local help **without making network requests**. It does not claim an authenticated complete remote listing. Downloading is disabled.

In [29]:
# Inspect installed CLI capabilities locally only; no network calls are made.
cli = shutil.which('kaggle')
if cli:
    for command in ([cli, 'competitions', 'files', '--help'], [cli, 'competitions', '--help']):
        result = subprocess.run(command, capture_output=True, text=True, timeout=15)
        print(' '.join(command))
        print((result.stdout or result.stderr)[:4000])
else:
    print('Kaggle CLI not installed. Remote listing is not attempted.')
print('Remote image inventory verified: False')

/usr/local/bin/kaggle competitions files --help
usage: kaggle competitions files [-h] [-v] [-q] [--page-token PAGE_TOKEN]
                                 [--page-size PAGE_SIZE]
                                 [competition]

options:
  -h, --help            show this help message and exit
  competition           Competition URL suffix (use "kaggle competitions list" to show options)
                        If empty, the default competition will be used (use "kaggle config set competition")"
  -v, --csv             Print results in CSV format (if not set print in table format)
  -q, --quiet           Suppress printing information about the upload/download progress
  --page-token PAGE_TOKEN
                        Page token for results paging.
  --page-size PAGE_SIZE
                        Number of items to show on a page. Default size is 20, max is 200

/usr/local/bin/kaggle competitions --help
usage: kaggle competitions [-h]
                           {list,files,download,submit,s

In [30]:
import re

images_dir = DATA_DIR / 'images'
VALID_IMAGE_SUFFIXES = {'.jpg', '.jpeg'}

# Inventory enumerates all local files, never an inferred/predicted path.
image_files = sorted(p for p in images_dir.rglob('*') if p.is_file()) if images_dir.is_dir() else []
local_relpaths = {p.relative_to(DATA_DIR).as_posix() for p in image_files}

archive_relpaths = None
archive_check = 'NO_ARCHIVE'
if IMAGE_ARCHIVE is not None:
    archive_path = Path(IMAGE_ARCHIVE)
    if archive_path.is_file() and zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path) as zf:
            all_members = [v for v in zf.infolist() if not v.is_dir()]
            image_members = [m.filename.lstrip('./') for m in all_members if '/images/' in ('/' + m.filename) or m.filename.startswith('images/')]
            # Strip an optional top-level enclosing folder from archive paths.
            archive_relpaths = set()
            for member in image_members:
                position = member.find('images/')
                archive_relpaths.add(member[position:])
        archive_check = 'MATCH' if archive_relpaths == local_relpaths else 'MISMATCH'
    else:
        archive_check = 'INVALID_OR_MISSING_ARCHIVE'

if archive_check == 'MATCH':
    image_coverage_status = 'VERIFIED_AGAINST_COMPLETE_PROVIDED_ARCHIVE'
elif images_dir.is_dir() and CONFIRM_LOCAL_IMAGE_INVENTORY_COMPLETE:
    image_coverage_status = 'USER_ATTESTED_COMPLETE_LOCAL_INVENTORY'
else:
    image_coverage_status = 'NOT_MEASURED'

print('Local image files:', f'{len(image_files):,}')
print('Archive comparison:', archive_check)
print('Image coverage status:', image_coverage_status)
if image_coverage_status == 'NOT_MEASURED':
    print('Exact coverage not measured. Confirm full local extraction, or provide an already downloaded complete archive.')

records = []
for p in image_files:
    rel = p.relative_to(DATA_DIR).as_posix()
    stem = p.stem
    valid_extension = p.suffix.lower() in VALID_IMAGE_SUFFIXES
    valid_stem = bool(re.fullmatch(r'[0-9]{10}', stem))
    records.append((rel, stem if valid_stem and valid_extension else None, valid_extension, valid_stem))

image_inventory = pd.DataFrame(records, columns=['relative_path','article_id','valid_extension','valid_stem'])
if image_inventory.empty:
    image_inventory = pd.DataFrame(columns=['relative_path','article_id','valid_extension','valid_stem'])

report('image_inventory_quality', pd.DataFrame([{
    'inventory_files': len(image_inventory),
    'valid_jpg_with_10digit_stem': int(image_inventory.article_id.notna().sum()),
    'invalid_extension_count': int((~image_inventory.valid_extension).sum()),
    'invalid_stem_count': int((~image_inventory.valid_stem).sum()),
    'duplicate_valid_stems': int(image_inventory.article_id.dropna().duplicated().sum()),
    'coverage_status': image_coverage_status,
    'archive_comparison': archive_check,
}]))

Local image files: 105,100
Archive comparison: NO_ARCHIVE
Image coverage status: NOT_MEASURED
Exact coverage not measured. Confirm full local extraction, or provide an already downloaded complete archive.
[image_inventory_quality] 1 aggregate rows; saved to /content/eda_outputs/image_inventory_quality.csv


,inventory_files,valid_jpg_with_10digit_stem,invalid_extension_count,invalid_stem_count,duplicate_valid_stems,coverage_status,archive_comparison
0,105100,105100,0,0,0,NOT_MEASURED,NO_ARCHIVE


,inventory_files,valid_jpg_with_10digit_stem,invalid_extension_count,invalid_stem_count,duplicate_valid_stems,coverage_status,archive_comparison
0,105100,105100,0,0,0,NOT_MEASURED,NO_ARCHIVE


### Exact image presence rates — gated

Computations below run **only** when the inventory completeness gate is satisfied. Counting filenames does not validate image bytes, decode success, or embedding coverage.

In [31]:
if image_coverage_status != 'NOT_MEASURED':
    image_ids = pd.DataFrame({'article_id': image_inventory.article_id.dropna().unique()})
    image_counts = image_inventory.dropna(subset=['article_id']).groupby('article_id').size().reset_index(name='image_files')
    con.register('image_id_inventory', image_ids)
    con.register('image_count_inventory', image_counts)

    report('image_coverage_overall', query("""
        WITH catalog AS (SELECT DISTINCT article_id FROM articles_raw WHERE article_id IS NOT NULL AND TRIM(article_id)<>''),
             matched AS (
              SELECT c.article_id, COALESCE(i.image_files, 0) AS image_files
              FROM catalog c LEFT JOIN image_count_inventory i USING(article_id)
             )
        SELECT COUNT(*) AS catalog_articles,
               COUNT(*) FILTER (WHERE image_files > 0) AS articles_with_image,
               COUNT(*) FILTER (WHERE image_files = 0) AS articles_without_image,
               ROUND(100.0 * COUNT(*) FILTER (WHERE image_files > 0) / NULLIF(COUNT(*),0), 4) AS article_image_coverage_pct,
               COUNT(*) FILTER (WHERE image_files > 1) AS articles_with_multiple_images,
               (SELECT COUNT(*) FROM image_count_inventory) AS unique_valid_image_stems,
               (SELECT COALESCE(SUM(image_files),0) FROM image_count_inventory) AS valid_image_files,
               (SELECT COUNT(*) FROM image_id_inventory img LEFT JOIN catalog c USING(article_id) WHERE c.article_id IS NULL) AS image_ids_absent_from_catalog
        FROM matched
    """))

    report('image_coverage_transaction_weighted', query("""
       SELECT COUNT(*) AS purchase_lines,
              COUNT(*) FILTER (WHERE i.article_id IS NOT NULL) AS purchase_lines_with_image,
              ROUND(100.0 * COUNT(*) FILTER (WHERE i.article_id IS NOT NULL) / NULLIF(COUNT(*),0), 4) AS transaction_line_weighted_image_coverage_pct
       FROM transactions_raw t LEFT JOIN image_id_inventory i USING(article_id)
    """))

    for category in ('product_group_name', 'product_type_name', 'section_name'):
        report('image_coverage_by_' + category, query(f"""
          WITH catalog AS (
           SELECT article_id, ANY_VALUE({category}) AS category
           FROM articles_raw WHERE article_id IS NOT NULL GROUP BY article_id
          )
          SELECT COALESCE(category, 'MISSING') AS category,
                 COUNT(*) AS catalog_articles,
                 COUNT(*) FILTER (WHERE i.article_id IS NOT NULL) AS articles_with_image,
                 ROUND(100.0 * COUNT(*) FILTER (WHERE i.article_id IS NOT NULL) / NULLIF(COUNT(*),0), 4) AS image_coverage_pct
          FROM catalog c LEFT JOIN image_id_inventory i USING(article_id)
          GROUP BY 1 ORDER BY catalog_articles DESC
        """), 20)

    report('image_coverage_by_transaction_observed', query("""
       WITH observed AS (SELECT DISTINCT article_id FROM transactions_raw),
            catalog AS (SELECT DISTINCT article_id FROM articles_raw WHERE article_id IS NOT NULL)
       SELECT CASE WHEN o.article_id IS NULL THEN 'NOT_OBSERVED' ELSE 'OBSERVED' END AS transaction_status,
              COUNT(*) AS catalog_articles,
              COUNT(*) FILTER (WHERE i.article_id IS NOT NULL) AS articles_with_image,
              ROUND(100.0 * COUNT(*) FILTER (WHERE i.article_id IS NOT NULL) / NULLIF(COUNT(*),0), 4) AS image_coverage_pct
       FROM catalog c LEFT JOIN observed o USING(article_id)
       LEFT JOIN image_id_inventory i USING(article_id)
       GROUP BY 1 ORDER BY 1
    """))
else:
    print('Image coverage is NOT_MEASURED: no complete inventory was verified or explicitly attested.')
print('Filename presence does not imply decodability or embedding availability.')

Image coverage is NOT_MEASURED: no complete inventory was verified or explicitly attested.
Filename presence does not imply decodability or embedding availability.


### Optional bounded file decodability sample

This is deliberately separate from filename coverage. It does **not** establish full-catalog decode success.

In [32]:
RUN_IMAGE_DECODE_SAMPLE = False
IMAGE_DECODE_SAMPLE_SIZE = 100
if RUN_IMAGE_DECODE_SAMPLE and image_files:
    from PIL import Image
    valid_paths = [p for p in image_files if p.suffix.lower() in VALID_IMAGE_SUFFIXES]
    sample = [valid_paths[i] for i in np.linspace(0, len(valid_paths) - 1, min(IMAGE_DECODE_SAMPLE_SIZE, len(valid_paths)), dtype=int)]
    succeeded = 0
    for path in sample:
        try:
            with Image.open(path) as img:
                img.verify()
            succeeded += 1
        except Exception:
            pass
    report('image_decode_sample', pd.DataFrame([{'sampled_files': len(sample), 'decoded_successfully': succeeded, 'failed': len(sample) - succeeded}]))
else:
    print('Optional bounded image decode sample skipped.')

Optional bounded image decode sample skipped.


## 5. Compact EDA gate summary

In [33]:
summary = {
    'generated_at_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'data_dir': str(DATA_DIR),
    'articles': article_overview.to_dict('records')[0],
    'article_conflicting_ids': article_conflicts.to_dict('records')[0],
    'customers': customer_overview.to_dict('records')[0],
    'transactions': transaction_overview.to_dict('records')[0],
    'image_coverage_status': image_coverage_status,
    'image_inventory_files_on_disk': len(image_files),
    'image_archive_check': archive_check,
    'notes': [
        'No training, ranking or final split was performed.',
        'Transaction purchase lines were not deduplicated.',
        'Product metadata is treated as a provided snapshot, not verified historical state.',
        'Unpurchased items are unobserved, not confirmed negative examples.',
        'Image filename presence is separate from image decoding and embedding coverage.',
        'Customer identifiers and postal codes are not included in reports.',
    ],
}
(OUTPUT_DIR / 'eda_gate_summary.json').write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print('Aggregate reports:', OUTPUT_DIR)
print('Review this EDA before choosing item eligibility, time cutoffs or retrieval models.')

{
  "generated_at_utc": "2026-10-29T10:56:51Z",
  "data_dir": "/content/data",
  "articles": {
    "rows": 105542,
    "unique_article_ids": 105542,
    "null_or_blank_ids": 0,
    "nonunique_id_rows": 0,
    "exact_duplicate_rows": 0
  },
  "article_conflicting_ids": {
    "duplicate_id_count": 0,
    "extra_rows_for_duplicate_ids": 0.0,
    "conflicting_ids": 0
  },
  "customers": {
    "rows": 1371980,
    "unique_customer_ids": 1371980,
    "null_or_blank_ids": 0,
    "nonunique_id_rows": 0,
    "exact_duplicate_rows": 0
  },
  "transactions": {
    "purchase_lines": 31788324,
    "first_valid_date": "2018-09-20 00:00:00",
    "last_valid_date": "2020-09-22 00:00:00",
    "malformed_or_missing_dates": 0,
    "distinct_customers": 1362281,
    "distinct_articles": 104547,
    "distinct_customer_article_pairs": 27306439,
    "distinct_customer_article_day_keys": 28575395,
    "exact_duplicate_rows": 2974905
  },
  "image_coverage_status": "NOT_MEASURED",
  "image_inventory_files_on_d

/tmp/ipykernel_2729/1771232770.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at_utc': datetime.utcnow().isoformat(timespec='seconds') + 'Z',


## image coverage analysis

Configure image inventory source

In [34]:
from pathlib import Path
import duckdb
import pandas as pd
import json
import re
from collections import Counter

DATA_DIR = Path("/content/data")
OUTPUT_DIR = Path("/content/eda_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_DIR = DATA_DIR / "images"

# Set to True only if the complete Kaggle image dataset
# was downloaded, extracted, and verified without errors.
CONFIRM_COMPLETE_LOCAL_DOWNLOAD = True

assert IMAGE_DIR.is_dir(), f"Image directory not found: {IMAGE_DIR}"

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW articles_raw AS
SELECT *
FROM read_csv(
    '{(DATA_DIR / "articles.csv").as_posix()}',
    all_varchar = true
)
""")

con.execute(f"""
CREATE OR REPLACE VIEW transactions_raw AS
SELECT *
FROM read_csv(
    '{(DATA_DIR / "transactions_train.csv").as_posix()}',
    all_varchar = true
)
""")

print("Image inventory source:", IMAGE_DIR)
print("Completeness confirmation:", CONFIRM_COMPLETE_LOCAL_DOWNLOAD)

Image inventory source: /content/data/images
Completeness confirmation: True


Enumerate and validate image filenames

In [35]:
image_records = []
invalid_files = []

valid_extensions = {".jpg", ".jpeg"}
article_id_pattern = re.compile(r"^\d{10}$")

for path in IMAGE_DIR.rglob("*"):
    if not path.is_file():
        continue

    extension = path.suffix.lower()
    article_id = path.stem

    if extension not in valid_extensions:
        invalid_files.append({
            "path": str(path.relative_to(IMAGE_DIR)),
            "reason": "Invalid extension"
        })
        continue

    if not article_id_pattern.fullmatch(article_id):
        invalid_files.append({
            "path": str(path.relative_to(IMAGE_DIR)),
            "reason": "Invalid article ID format"
        })
        continue

    image_records.append({
        "article_id": article_id,
        "image_path": str(path.relative_to(IMAGE_DIR))
    })

images_df = pd.DataFrame(
    image_records,
    columns=["article_id", "image_path"]
)

invalid_df = pd.DataFrame(
    invalid_files,
    columns=["path", "reason"]
)

image_counts = images_df.groupby("article_id").size()

inventory_summary = {
    "total_image_files": len(images_df) + len(invalid_df),
    "valid_image_files": len(images_df),
    "invalid_files": len(invalid_df),
    "unique_image_article_ids": images_df["article_id"].nunique(),
    "articles_with_multiple_images": int((image_counts > 1).sum()),
    "duplicate_image_mappings": int(
        (image_counts - 1).clip(lower=0).sum()
    ),
}

print(json.dumps(inventory_summary, indent=2))

print("\nInvalid filename examples:")
display(invalid_df.head(10))

images_df.to_parquet(
    OUTPUT_DIR / "image_inventory.parquet",
    index=False
)

{
  "total_image_files": 105100,
  "valid_image_files": 105100,
  "invalid_files": 0,
  "unique_image_article_ids": 105100,
  "articles_with_multiple_images": 0,
  "duplicate_image_mappings": 0
}

Invalid filename examples:


,path,reason


Verify coverage against articles and transactions

In [36]:
con.register("image_inventory", images_df)

image_coverage_status = (
    "MEASURED_CONFIRMED_LOCAL_DOWNLOAD"
    if CONFIRM_COMPLETE_LOCAL_DOWNLOAD
    else "NOT_MEASURED"
)

coverage = con.execute("""
WITH image_ids AS (
    SELECT DISTINCT article_id
    FROM image_inventory
),
catalog AS (
    SELECT
        article_id,
        product_group_name,
        product_type_name,
        section_name
    FROM articles_raw
),
catalog_coverage AS (
    SELECT
        COUNT(*) AS total_articles,
        COUNT(*) FILTER (
            WHERE i.article_id IS NOT NULL
        ) AS articles_with_image,
        COUNT(*) FILTER (
            WHERE i.article_id IS NULL
        ) AS articles_without_image
    FROM catalog a
    LEFT JOIN image_ids i USING (article_id)
),
transaction_coverage AS (
    SELECT
        COUNT(*) AS total_transaction_lines,
        COUNT(*) FILTER (
            WHERE i.article_id IS NOT NULL
        ) AS transaction_lines_with_image
    FROM transactions_raw t
    LEFT JOIN image_ids i USING (article_id)
)
SELECT
    c.*,
    ROUND(
        100.0 * c.articles_with_image
        / NULLIF(c.total_articles, 0),
        4
    ) AS article_image_coverage_pct,
    t.total_transaction_lines,
    t.transaction_lines_with_image,
    ROUND(
        100.0 * t.transaction_lines_with_image
        / NULLIF(t.total_transaction_lines, 0),
        4
    ) AS transaction_weighted_image_coverage_pct
FROM catalog_coverage c
CROSS JOIN transaction_coverage t
""").df()

unmatched_images = con.execute("""
SELECT COUNT(*) AS image_ids_not_in_catalog
FROM (
    SELECT DISTINCT article_id
    FROM image_inventory
) i
ANTI JOIN articles_raw a
USING (article_id)
""").df()

print("Image coverage status:", image_coverage_status)

if CONFIRM_COMPLETE_LOCAL_DOWNLOAD:
    display(coverage)
    display(unmatched_images)
else:
    print(
        "Complete image coverage is not certified. "
        "The local inventory can be analyzed, but its completeness "
        "has not been confirmed."
    )

coverage.to_csv(
    OUTPUT_DIR / "image_coverage_summary.csv",
    index=False
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Image coverage status: MEASURED_CONFIRMED_LOCAL_DOWNLOAD


,total_articles,articles_with_image,articles_without_image,article_image_coverage_pct,total_transaction_lines,transaction_lines_with_image,transaction_weighted_image_coverage_pct
0,105542,105100,442,99.5812,31788324,31651678,99.5701


,image_ids_not_in_catalog
0,0


Image coverage by category

In [37]:
segment_queries = {
    "product_group_name": "product_group_name",
    "product_type_name": "product_type_name",
    "section_name": "section_name",
}

image_segments = {}

for segment_name, column in segment_queries.items():

    result = con.execute(f"""
    WITH image_ids AS (
        SELECT DISTINCT article_id
        FROM image_inventory
    )
    SELECT
        a.{column} AS segment,
        COUNT(*) AS total_articles,
        COUNT(*) FILTER (
            WHERE i.article_id IS NOT NULL
        ) AS articles_with_image,
        COUNT(*) FILTER (
            WHERE i.article_id IS NULL
        ) AS articles_without_image,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE i.article_id IS NOT NULL
            ) / NULLIF(COUNT(*), 0),
            3
        ) AS image_coverage_pct
    FROM articles_raw a
    LEFT JOIN image_ids i
        USING (article_id)
    GROUP BY 1
    ORDER BY total_articles DESC
    """).df()

    image_segments[segment_name] = result

    result.to_csv(
        OUTPUT_DIR / f"image_coverage_by_{segment_name}.csv",
        index=False
    )

    print(f"\nCoverage by {segment_name}")
    display(result.head(15))

# Compare purchased and never-purchased articles
observed_coverage = con.execute("""
WITH image_ids AS (
    SELECT DISTINCT article_id
    FROM image_inventory
),
purchased_ids AS (
    SELECT DISTINCT article_id
    FROM transactions_raw
)
SELECT
    CASE
        WHEN p.article_id IS NOT NULL
        THEN 'Transaction observed'
        ELSE 'Never observed'
    END AS transaction_status,
    COUNT(*) AS total_articles,
    COUNT(*) FILTER (
        WHERE i.article_id IS NOT NULL
    ) AS articles_with_image,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE i.article_id IS NOT NULL
        ) / NULLIF(COUNT(*), 0),
        3
    ) AS image_coverage_pct
FROM articles_raw a
LEFT JOIN image_ids i USING (article_id)
LEFT JOIN purchased_ids p USING (article_id)
GROUP BY 1
ORDER BY 1
""").df()

display(observed_coverage)

observed_coverage.to_csv(
    OUTPUT_DIR / "image_coverage_by_transaction_status.csv",
    index=False
)

# Save the final image audit summary
image_audit_summary = {
    "image_coverage_status": image_coverage_status,
    "inventory": inventory_summary,
    "completeness_basis": (
        "User-confirmed complete local Kaggle extraction"
        if CONFIRM_COMPLETE_LOCAL_DOWNLOAD
        else "Local inventory completeness not verified"
    ),
    "decode_validation": "NOT_PERFORMED",
}

with open(OUTPUT_DIR / "image_audit_summary.json", "w") as file:
    json.dump(image_audit_summary, file, indent=2)

print("\nImage audit outputs saved to:", OUTPUT_DIR)


Coverage by product_group_name


,segment,total_articles,articles_with_image,articles_without_image,image_coverage_pct
0,Garment Upper body,42741,42680,61,99.857
1,Garment Lower body,19812,19770,42,99.788
2,Garment Full body,13292,13276,16,99.880
3,Accessories,11158,11005,153,98.629
4,Underwear,5490,5462,28,99.490
5,Shoes,5283,5156,127,97.596
6,Swimwear,3127,3125,2,99.936
7,Socks & Tights,2442,2431,11,99.550
8,Nightwear,1899,1898,1,99.947
9,Unknown,121,121,0,100.000



Coverage by product_type_name


,segment,total_articles,articles_with_image,articles_without_image,image_coverage_pct
0,Trousers,11169,11150,19,99.830
1,Dress,10362,10348,14,99.865
2,Sweater,9302,9295,7,99.925
3,T-shirt,7904,7880,24,99.696
4,Top,4155,4152,3,99.928
5,Blouse,3979,3975,4,99.899
6,Jacket,3940,3940,0,100.000
7,Shorts,3939,3926,13,99.670
8,Shirt,3405,3403,2,99.941
9,Vest top,2991,2979,12,99.599



Coverage by section_name


,segment,total_articles,articles_with_image,articles_without_image,image_coverage_pct
0,Womens Everyday Collection,7295,7281,14,99.808
1,Divided Collection,7124,7115,9,99.874
2,Baby Essentials & Complements,4932,4916,16,99.676
3,Kids Girl,4469,4466,3,99.933
4,Young Girl,3899,3894,5,99.872
5,Womens Lingerie,3598,3581,17,99.528
6,Girls Underwear & Basics,3490,3483,7,99.799
7,Womens Tailoring,3376,3372,4,99.882
8,Kids Boy,3328,3325,3,99.910
9,Womens Small accessories,3270,3208,62,98.104


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transaction_status,total_articles,articles_with_image,image_coverage_pct
0,Never observed,995,994,99.899
1,Transaction observed,104547,104106,99.578



Image audit outputs saved to: /content/eda_outputs
